# Оценка релевантности организаций запросам на Яндекс.Картах с помощью LLM-агента

<img src="https://sun9-65.userapi.com/impg/N4y2cxlL7PauAs82tBNFOUAiNctFICWDy4Mbiw/Jiz1fb7NLWU.jpg?size=1080x1080&quality=95&sign=df2786058624d9ccac3ede4d5d056e2f&type=album" width="500" height="500" />


## Описание и загрузка данных

Данные: https://disk.yandex.ru/d/6d5hFHvpAZjQdw

In [1]:
import requests

public_url = "https://disk.yandex.ru/d/6d5hFHvpAZjQdw"  # твоя публичная ссылка
api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"

resp = requests.get(api_url, params={"public_key": public_url})
resp.raise_for_status()
download_url = resp.json()["href"]  # это уже прямая ссылка на файл

dest = "/content/data.jsonl"
with requests.get(download_url, stream=True) as r:
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)


In [2]:
import json
import pandas as pd

records = []
with open("/content/data.jsonl", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        if i == 2659:      # пропускаем битую строку
            continue
        try:
            obj = json.loads(line)
            records.append(obj)
        except Exception as e:
            print("ещё битая строка:", i, e)

data = pd.DataFrame(records)


In [3]:
data['relevance'].unique()

array([1. , 0. , 0.1])

In [4]:
data['relevance'].value_counts()

,count
relevance,
1.0,15881
0.0,14509
0.1,4703


Здесь 1.0 соответствует оценке RELEVANT_PLUS, 0.1 -- оценке RELEVANT_MINUS, 0.0 -- оценке IRRELEVANT.

Ваша задача -- построить LLM-агента, который будет предсказывать релевантность.

Выделим данные для оценки качества агента. Запуск агента -- это тяжелая и потенциально дорогая операция. Поэтому eval-множество имеет размер 500. Также для простоты из eval-множества выкинуты данные с оценкой RELEVANT_MINUS. Тем не менее, вы можете использовать такие примеры для подачи примеров агенту.

**ОБРАТИТЕ ВНИМАНИЕ, ЧТО В EVAL-ДАННЫЕ НЕЛЬЗЯ ПОДГЛЯДЫВАТЬ ДЛЯ КАЛИБРОВКИ АГЕНТА!!! ДЛЯ ЭТОГО ЕСТЬ ОБУЧАЮЩИЕ ДАННЫЕ**

В качестве метрики качества мы будем использовать обычную ACCURACY, поскольку классы сбалансированы.

In [5]:
train_data = data[570:]
eval_data = data[:570]
eval_data = eval_data[eval_data["relevance"] != 0.1]
eval_data

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,relevance,reviews_summarized
0,сигары,"Москва, Дубравная улица, 34/29",Tabaccos; Магазин Tabaccos; Табаккос,Магазин табака и курительных принадлежностей,1263329400,None,1.0,"Организация занимается продажей табака, курите..."
1,кальянная спб мероприятия,"Санкт-Петербург, Большой проспект Петроградско...",PioNero; Pionero; Пицца Паста бар; Pio Nero; P...,Кафе,228111266197,PioNero предлагает разнообразные блюда итальян...,0.0,"Организация PioNero — это кафе, бар и ресторан..."
2,Эпиляция,"Московская область, Одинцово, улица Маршала Жу...",MaxiLife; Центр красоты и здоровья MaxiLife; Ц...,Стоматологическая клиника,1247255817,"Стоматологическая клиника, массажный салон и к...",1.0,"Организация занимается стоматологическими, кос..."
4,стиральных машин,"Москва, улица Обручева, 34/63",М.Видео; M Video; M. Видео; M.Видео; Mvideo; М...,Магазин бытовой техники,1074529324,М.Видео предлагает широкий ассортимент бытовой...,1.0,Организация занимается продажей бытовой техник...
5,сеть быстрого питания,"Санкт-Петербург, 1-я Красноармейская улица, 15",Rostic's; KFC; Ресторан быстрого питания KFC,Быстрое питание,1219173871,Rostic's предлагает различные наборы быстрого ...,1.0,"Организация занимается быстрым питанием, предо..."
...,...,...,...,...,...,...,...,...
561,наращивание ресниц,"Саратов, улица имени А.С. Пушкина, 1",Сила; Sila; Beauty brow; Студия бровей Beauty ...,Салон красоты,236976975812,Салон красоты «Сила» предлагает услуги по уход...,1.0,Организация «Сила» занимается предоставлением ...
565,игры,"Москва, Щёлковское шоссе, 79, корп. 1",YouPlay; YouPlay КиберКлуб,Компьютерный клуб,109673025161,YouPlay КиберКлуб предлагает услуги по игре на...,0.0,Организация занимается предоставлением услуг к...
566,домашний интернет в курске что подключить отзы...,"Курск, Садовая улица, 5",Цифровой канал; Digital Channel; DChannel; ЦК;...,Телекоммуникационная компания,1737991898,None,0.0,None
567,гостиница волгодонск сауна номер телефона,"Ростовская область, городской округ Волгодонск...",Поплавок; Poplavok,"База , дом отдыха",147783493467,"Предлагает размещение в различных типах жилья,...",0.0,Организация «Поплавок» предлагает услуги базы ...


In [6]:
eval_data.to_excel("eval_data.xlsx")

# Импорты

In [8]:
!pip install -q langchain langgraph langchain_openai langchain_core langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [80]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END, MessagesState
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.tools import TavilySearchResults
from langchain_core.tools import tool

import json
import re
from typing import TypedDict, Dict, Any, List, Optional, Literal
import os
import time, uuid
import math
from dotenv import load_dotenv

In [81]:
load_dotenv()

True

# Безйлайн: вызов llm без поисковика

In [75]:
BASELINE_SYSTEM_PROMPT = """
Ты — строгий бинарный классификатор релевантности организации рубричному запросу.

У тебя есть только:
- query: запрос пользователя
- card_text: карточка организации (описание/рубрика/услуги/цены/отзывы)

ПРАВИЛА (строго):
1) Если запрос точно подходит к ЛЮБОЙ организации этой рубрики, то Ответ = 1.
2) Ответ = 1 ТОЛЬКО если в card_text есть подтверждение запроса, то есть
прямое упоминание требуемой услуги/товара/типа места/атрибута
3) Если в запросе есть обязательные ограничения (район, режим работы, конкретная услуга, и т.п.), то 1 можно ставить только если эти ограничения
   ЯВНО подтверждены в card_text.
4) Субъективные оценки, например: (уютный, романтичный, недорого, лучший) можно подтверждать только
   если в отзывах/описании есть упоминания этого качества. Иначе 0.

ФОРМАТ ОТВЕТА (строго):
- Верни ровно один символ: 0 или 1.
- Никаких пояснений, никаких пробелов, никаких JSON, никаких переносов строк.
""".strip()



In [76]:
def _clean_str(x) -> str:
    if x is None:
        return ""
    try:
        if isinstance(x, float) and math.isnan(x):
            return ""
    except Exception:
        pass
    s = str(x).strip()
    return "" if s.lower() in {"nan", "none"} else s

def make_card_text(row: Dict[str, Any], max_reviews_len: int = 900) -> str:
    """
    Делает "card_text" из реальных полей карточки.
    ВАЖНО: никакого relevance/label.
    """
    name = _clean_str(row.get("name", ""))
    address = _clean_str(row.get("address", ""))
    rubric = _clean_str(row.get("normalized_main_rubric_name_ru", row.get("rubric", "")))
    permalink = _clean_str(row.get("permalink", ""))
    prices = _clean_str(row.get("prices_summarized", ""))
    reviews = _clean_str(row.get("reviews_summarized", ""))

    lines = ["Информация об организации:"]
    if name: lines.append(f"- Название: {name}")
    if rubric: lines.append(f"- Рубрика: {rubric}")
    if address: lines.append(f"- Адрес: {address}")
    if permalink: lines.append(f"- Permalink: {permalink}")
    if prices: lines.append(f"- Услуги/цены (summary): {prices}")
    if reviews:
        lines.append(f"- Отзывы (summary): {reviews[:max_reviews_len]}")

    return "\n".join(lines).strip()


In [77]:
import re
from langchain_core.messages import SystemMessage, HumanMessage

def baseline_predict(row: dict) -> int:
    query = str(row.get("Text", "") or "").strip()
    card_text = make_card_text(row, max_reviews_len=1500)

    payload = f"query: {query}\n\ncard_text:\n{card_text}"

    resp = llm.invoke([
        SystemMessage(content=BASELINE_SYSTEM_PROMPT),
        HumanMessage(content=payload)
    ])
    out = (resp.content or "").strip()
    # Строгая нормализация: берём первую цифру 0/1, иначе 0
    m = re.search(r"[01]", out)
    return int(m.group(0)) if m else 0


In [78]:
import random

def to_true_label(rel) -> int:
    try:
        v = float(rel)
    except Exception:
        return 0
    return 1 if v >= 0.9 else 0


def evaluate_baseline(train_data, n=150, seed=42):
    random.seed(seed)
    idxs = list(range(len(train_data)))
    random.shuffle(idxs)

    sample_idxs = []
    for idx in idxs:
        rel = train_data.iloc[idx].get("relevance", None)
        try:
            if float(rel) == 0.1:
                continue
        except Exception:
            pass
        sample_idxs.append(idx)
        if len(sample_idxs) == n:
            break

    assert len(sample_idxs) == n, f"Не смог набрать {n} примеров без relevance=0.1"

    TP = FP = TN = FN = 0
    logs = []

    for k, idx in enumerate(sample_idxs, start=1):
        row = train_data.iloc[idx].to_dict()
        true = to_true_label(row.get("relevance", 0.0))
        pred = baseline_predict(row)

        if true == 1 and pred == 1: TP += 1
        elif true == 0 and pred == 1: FP += 1
        elif true == 0 and pred == 0: TN += 1
        else: FN += 1

        logs.append({"idx": idx, "true": true, "pred": pred, "Text": row.get("Text","")})
        total = TP + FP + TN + FN
        acc = (TP + TN) / total if total else 0.0
        precision = TP / (TP + FP) if (TP + FP) else 0.0
        recall = TP / (TP + FN) if (TP + FN) else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

        print(
            f"[{k}/{n}] "
            f"acc={acc:.4f} precision={precision:.4f} recall={recall:.4f} f1={f1:.4f} "
            f"| TP={TP} FP={FP} TN={TN} FN={FN}"
        )

    total = TP + FP + TN + FN
    accuracy = (TP + TN) / total if total else 0.0
    precision = TP / (TP + FP) if (TP + FP) else 0.0
    recall = TP / (TP + FN) if (TP + FN) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    print("CONFUSION MATRIX")
    print("             Pred=1  Pred=0")
    print(f"True=1        {TP:5d}  {FN:5d}")
    print(f"True=0        {FP:5d}  {TN:5d}")
    print()
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1:        {f1:.4f}")

    return {
        "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1,
        "logs": logs,
        "sample_idxs": sample_idxs
    }

# запуск
res = evaluate_baseline(train_data, n=150, seed=42)


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1769472000000'}, 'provider_name': None}}, 'user_id': 'user_35lMavUOjIClZzhxsoLbmYABpHr'}

В итоге с различными промптами качество безйлайна получилось не выше чем:

Accuracy:  0.6600

F1:        0.7151

Precision: 0.6957

Recall:    0.7356